# 02. Trip Operational Dataset

## Objective

Build a trip-level operational dataset by joining multiple source tables and calculating base operational metrics.

Each row represents one trip and serves as the operational input for downstream analytical processing.

At this stage the dataset includes:

- trip attributes
- client, route, truck and driver information
- calculated revenue
- estimated fuel consumption (liters)

Fuel cost allocation, FIFO pricing and profitability metrics are intentionally excluded from this notebook and will be calculated during the analytical layer.

In [10]:
import os
from pathlib import Path

import pandas as pd

from dotenv import load_dotenv
from sqlalchemy import create_engine, text

In [11]:
# Load environment variables
load_dotenv()

DB_USER = os.getenv("DB_USER")
DB_PASSWORD = os.getenv("DB_PASSWORD")
DB_HOST = os.getenv("DB_HOST")
DB_PORT = os.getenv("DB_PORT")
DB_NAME = os.getenv("DB_NAME")

engine = create_engine(
    f"mysql+pymysql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}",
    pool_pre_ping=True,
    future=True
)

try:
    with engine.connect() as connection:
        connection.execute(text("SELECT 1"))
    print("✅ Successfully connected to MySQL.")
except Exception as e:
    print(f"❌ Connection failed:\n{e}")

✅ Successfully connected to MySQL.


In [12]:
sql_path = Path("../sql/04_trip_operational_dataset.sql")

with open(sql_path, "r", encoding="utf-8") as file:
    query = file.read()

In [13]:
trip_operational_dataset = pd.read_sql(
    text(query),
    engine
)

print(f"Dataset successfully loaded.")

trip_operational_dataset.head()

Dataset successfully loaded.


,trip_id,date_departure,date_arrival,status,client_id,client_name,client_region,route_id,origin_city,destination_city,...,truck_capacity_tons,truck_year,driver_id,driver_name,driver_commission_rate,cargo_tons_actual,load_factor_pct,delay_hours,revenue_uah,estimated_fuel_liters
0,TRP-00001,2022-07-01,2022-07-02,completed,CLT-02,ПП ДніпроЗерно,Дніпропетровська,RTE-02,Дніпро,Одеса,...,24.0,2018,DRV-04,Андрій Савченко,0.13,20.76,86.5,1.8,18715.5552,239.59
1,TRP-00002,2022-07-01,2022-07-01,completed,CLT-09,ФГ Золоте Колосся,Хмельницька,RTE-09,Хмельницький,Тернопіль,...,24.0,2018,DRV-04,Андрій Савченко,0.13,16.44,68.5,1.5,3829.8624,56.85
2,TRP-00003,2022-07-01,2022-07-01,completed,CLT-09,ФГ Золоте Колосся,Хмельницька,RTE-09,Хмельницький,Тернопіль,...,24.5,2018,DRV-05,Ігор Шевченко,0.13,16.74,68.3,1.1,3899.7504,54.93
3,TRP-00004,2022-07-01,2022-07-01,delayed,CLT-08,ТОВ Пшеничний Шлях,Чернігівська,RTE-08,Чернігів,Київ,...,24.5,2018,DRV-11,Тарас Гончар,0.13,24.47,99.9,2.1,6526.3937,73.07
4,TRP-00005,2022-07-01,2022-07-01,completed,CLT-07,ФГ Жнива-Поділля,Вінницька,RTE-07,Вінниця,Жмеринка,...,24.5,2018,DRV-05,Ігор Шевченко,0.13,19.47,79.5,0.4,1883.1384,25.50


### Quick Quality Check

In [14]:
print(f"Rows: {len(trip_operational_dataset):,}")
print(f"Columns: {trip_operational_dataset.shape[1]}")

Rows: 4,925
Columns: 28


In [19]:
print(
    f"Date range: "
    f"{trip_operational_dataset['date_departure'].min()} "
    f"— "
    f"{trip_operational_dataset['date_departure'].max()}"
)

Date range: 2022-07-01 00:00:00 — 2024-10-31 00:00:00


In [15]:
trip_operational_dataset.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4925 entries, 0 to 4924
Data columns (total 28 columns):
 #   Column                  Non-Null Count  Dtype         
---  ------                  --------------  -----         
 0   trip_id                 4925 non-null   object        
 1   date_departure          4925 non-null   datetime64[ns]
 2   date_arrival            4925 non-null   object        
 3   status                  4925 non-null   object        
 4   client_id               4925 non-null   object        
 5   client_name             4925 non-null   object        
 6   client_region           4925 non-null   object        
 7   route_id                4925 non-null   object        
 8   origin_city             4925 non-null   object        
 9   destination_city        4925 non-null   object        
 10  route_type              4925 non-null   object        
 11  cargo_type              4925 non-null   object        
 12  trip_distance_km        4925 non-null   float64 

In [16]:
trip_operational_dataset.describe()

,date_departure,trip_distance_km,route_distance_km,route_has_port,truck_capacity_tons,truck_year,driver_commission_rate,cargo_tons_actual,load_factor_pct,delay_hours,revenue_uah,estimated_fuel_liters
count,4925,4925.000000,4925.000000,4925.000000,4925.000000,4925.000000,4925.000000,4925.000000,4925.000000,4925.000000,4880.000000,4827.000000
mean,2023-08-08 21:09:49.888325120,273.460508,273.460508,0.415838,23.958985,2018.836548,0.126435,21.809637,91.026924,3.337096,12083.252492,95.596967
min,2022-07-01 00:00:00,52.000000,52.000000,0.000000,21.500000,2015.000000,0.120000,15.000000,68.000000,0.000000,1513.200000,12.480000
25%,2022-11-29 00:00:00,132.000000,132.000000,0.000000,23.500000,2018.000000,0.120000,20.410000,87.200000,0.500000,6303.832400,46.560000
50%,2023-08-07 00:00:00,246.000000,246.000000,0.000000,24.000000,2019.000000,0.130000,22.050000,91.900000,1.000000,11689.725600,80.710000
75%,2024-03-23 00:00:00,472.000000,472.000000,1.000000,25.000000,2020.000000,0.130000,23.350000,96.600000,1.600000,16507.775900,134.620000
max,2024-10-31 00:00:00,612.000000,612.000000,1.000000,25.000000,2021.000000,0.130000,29.000000,117.400000,59.600000,29550.236400,510.430000
std,NaN,173.895158,173.895158,0.492916,1.032004,1.760971,0.004790,2.465021,9.492095,8.067492,6735.052183,61.591481


In [17]:
trip_operational_dataset.isna().sum()

trip_id                    0
date_departure             0
date_arrival               0
status                     0
client_id                  0
client_name                0
client_region              0
route_id                   0
origin_city                0
destination_city           0
route_type                 0
cargo_type                 0
trip_distance_km           0
route_distance_km          0
route_has_port             0
truck_id                   0
truck_brand                0
truck_model                0
truck_capacity_tons        0
truck_year                 0
driver_id                  0
driver_name                0
driver_commission_rate     0
cargo_tons_actual          0
load_factor_pct            0
delay_hours                0
revenue_uah               45
estimated_fuel_liters     98
dtype: int64

In [18]:
print("Missing revenue:")
display(
    trip_operational_dataset.loc[
        trip_operational_dataset["revenue_uah"].isna(),
        ["status"]
    ].value_counts()
)

print("\nMissing estimated fuel liters:")
display(
    trip_operational_dataset.loc[
        trip_operational_dataset["estimated_fuel_liters"].isna(),
        ["status"]
    ].value_counts()
)

Missing revenue:


status   
cancelled    45
Name: count, dtype: int64


Missing estimated fuel liters:


status   
cancelled    45
completed    43
delayed      10
Name: count, dtype: int64

### QA Notes

- Revenue is missing only for cancelled trips by design.
- Estimated fuel consumption is missing for cancelled trips and for trips occurring after the last recorded refueling event, since fuel consumption cannot yet be closed within the available observation window.


### Save dataset

In [21]:
output_path = Path("../data/processed/trip_operational_dataset.csv")

trip_operational_dataset.to_csv(
    output_path,
    index=False
)

print(f"✅ Dataset saved to:\n{output_path}")

✅ Dataset saved to:
../data/processed/trip_operational_dataset.csv


## Summary

The operational trip-level dataset has been successfully created.

Output:

`data/processed/trip_operational_dataset.csv`

This dataset becomes the operational source for the next pipeline step, where FIFO fuel allocation and trip-level profitability metrics will be calculated.